## [SQL] PythonでのSQL実行環境と基本的なSELECT


データ分析やシステム開発で必須となる **SQL (Structured Query Language)** のテクニックを、実例を交えて自分用にメモしおく。

今回は、本シリーズで使用する **Python上でSQLを実行・検証する環境** の構築と、最も基本的な `SELECT` 文について解説する。

本シリーズでは、軽量で扱いやすい **SQLite** と、高速なデータフレームライブラリ **Polars** を組み合わせて使用する。

## 準備

本サイトではpythonのライブラリであるsqlite3とipython-sqlを利用して、Jupyter Notebook上でSQLを実行・解説する。
また、データフレームの表示には **Polars** を利用する。

### github
- githubのjupyter notebook形式のファイルは[こちら](https://github.com/hiroshi0530/wa-src/blob/master/article/library/sql/01/01_nb.md)

### google colaboratory
- google colaboratory で実行する場合は[こちら](https://colab.research.google.com/github/hiroshi0530/wa-src/blob/master/article/library/sql/01/01_nb.ipynb)

### 筆者の環境
筆者の環境である。

In [1]:
!sw_vers

ProductName:		macOS
ProductVersion:		15.5
BuildVersion:		24F74


In [2]:
!python -V

Python 3.12.12


必要なライブラリを読み込む。

In [3]:
import sqlite3
import polars as pl

%load_ext sql
%config SqlMagic.feedback = True

%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [ ]:
## データの準備

まずは、SQLを実行するためのデータベースファイル（SQLite）を作成し、サンプルデータとして `Users` テーブルを作成する。

In [4]:
import os

if os.path.exists("data.db"):
    os.remove("data.db")

os.system("touch data.db")
os.system("chmod 664 data.db")

# データベース接続
%sql sqlite:///data.db

'Connected: @data.db'

In [5]:
%%sql
CREATE TABLE Users (
    id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    country TEXT
);

INSERT INTO Users (name, age, country) VALUES
    ('Alice', 25, 'USA'),
    ('Bob', 30, 'UK'),
    ('Charlie', 35, 'USA'),
    ('Dave', 20, 'Japan'),
    ('Eve', 28, 'Japan');

 * sqlite:///data.db
Done.
5 rows affected.


[]

## SQLの実行と結果の確認

本シリーズでは、SQLの実行結果を **Polars** の DataFrame として受け取り、綺麗に表示する方法を採用する。
`sqlite3` で接続し、`pl.read_database` を使うことで、SQLの結果を直接 DataFrame に変換できる。

### 基本的なSELECT

まずはテーブルの中身をすべて取得してみる。

In [6]:
query = """
SELECT * FROM Users;
"""

with sqlite3.connect("data.db") as conn:
    df = pl.read_database(query, connection=conn)
    display(df)

id,name,age,country
i64,str,i64,str
1,"""Alice""",25,"""USA"""
2,"""Bob""",30,"""UK"""
3,"""Charlie""",35,"""USA"""
4,"""Dave""",20,"""Japan"""
5,"""Eve""",28,"""Japan"""


### 条件指定 (WHERE) と 並べ替え (ORDER BY)

次に、条件を指定してデータを絞り込み、並べ替えてみる。
例：`country` が 'Japan' のユーザーを、`age` の降順（大きい順）で取得する。

In [7]:
query = """
SELECT
    name,
    age,
    country
FROM
    Users
WHERE
    country = 'Japan'
ORDER BY
    age DESC;
"""

with sqlite3.connect("data.db") as conn:
    df = pl.read_database(query, connection=conn)
    display(df)

name,age,country
str,i64,str
"""Eve""",28,"""Japan"""
"""Dave""",20,"""Japan"""


## まとめ

今回は、本シリーズで使用するSQL実行環境のセットアップと、基本的なクエリの実行方法を紹介した。

- **SQLite**: 手軽に使えるファイル型データベース。
- **Polars**: 高速で使いやすいデータフレームライブラリ。SQLの結果表示にも便利。

次回以降は、この環境を使って、より実践的なSQLのテクニック（Window関数、CTE、結合など）を解説していく。